# 3. Train Baseline Model (LSTM Seq2Seq + Attention)

**Chiến lược:** Mỗi epoch chỉ train trên một **subset nhỏ** (~500k câu).  
→ Epoch ngắn → Checkpoint lưu thường xuyên → An toàn khi mất kết nối.  
→ Hỗ trợ **Auto-Resume** khi Colab bị ngắt.  
→ Biểu đồ chỉ vẽ **1 lần duy nhất sau khi train xong**.

In [ ]:
!pip install tokenizers torch matplotlib -q

In [ ]:
import torch
import torch.nn as nn
import os, json, random
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn.utils.rnn import pad_sequence
from tokenizers import Tokenizer
from torch.cuda.amp import GradScaler, autocast

# --- Kết nối Google Drive ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/Multilingual_MT'
    print('✅ Google Colab + Drive đã kết nối.')
except:
    BASE_DIR = '/kaggle/input/multilingual-mt-data'
    print('⚠️ Kaggle mode.')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## ⚙️ Siêu tham số (Chỉnh ở đây)

In [ ]:
# ============================================================
#  SIÊU THAM SỐ
# ============================================================
EPOCHS       = 30        # Nhiều epoch, mỗi epoch ngắn
SUBSET_SIZE  = 500_000   # Số câu mỗi epoch
BATCH_SIZE   = 64
ACCUM_STEPS  = 4         # Effective batch = 256
LR           = 1e-3
MAX_LEN      = 128
EMBED_SIZE   = 256
HIDDEN_SIZE  = 512
NUM_LAYERS   = 2

OUT_DIR = f"{BASE_DIR}/model_assets"
os.makedirs(OUT_DIR, exist_ok=True)

steps_per_epoch = (SUBSET_SIZE // BATCH_SIZE) // ACCUM_STEPS
print(f'Steps/epoch (ước lượng): {steps_per_epoch:,}')
print(f'Tổng steps cả khóa train: {steps_per_epoch * EPOCHS:,}')

## Kiến trúc LSTM Seq2Seq + Luong Attention

In [ ]:
class EncoderLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers,
                            bidirectional=True, batch_first=True, dropout=0.3)
        self.fc_hidden = nn.Linear(hidden_size * 2, hidden_size)
        self.fc_cell   = nn.Linear(hidden_size * 2, hidden_size)

    def forward(self, x):
        outputs, (hidden, cell) = self.lstm(self.embedding(x))
        hidden = self.fc_hidden(torch.cat((hidden[0:1], hidden[1:2]), dim=2))
        cell   = self.fc_cell(torch.cat((cell[0:1], cell[1:2]), dim=2))
        return outputs, hidden, cell

class DecoderLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=pad_idx)
        self.lstm    = nn.LSTM(embed_size + hidden_size * 2, hidden_size, num_layers, batch_first=True)
        self.fc_out  = nn.Linear(hidden_size, vocab_size)
        self.energy  = nn.Linear(hidden_size * 3, 1)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x, hidden, cell, enc_out):
        embedded = self.embedding(x.unsqueeze(1))
        h = hidden.repeat(enc_out.shape[1], 1, 1).transpose(0, 1)
        attention = self.softmax(self.energy(torch.cat((h, enc_out), dim=2)))
        context   = torch.bmm(attention.transpose(1, 2), enc_out)
        output, (hidden, cell) = self.lstm(torch.cat((context, embedded), dim=2), (hidden, cell))
        return self.fc_out(output.squeeze(1)), hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing=0.5):
        enc_out, hidden, cell = self.encoder(src)
        vocab_size = self.decoder.fc_out.out_features
        outputs    = torch.zeros(src.shape[0], tgt.shape[1], vocab_size).to(src.device)
        x = tgt[:, 0]
        for t in range(1, tgt.shape[1]):
            output, hidden, cell = self.decoder(x, hidden, cell, enc_out)
            outputs[:, t] = output
            x = tgt[:, t] if random.random() < teacher_forcing else output.argmax(1)
        return outputs

## Dataset (đọc 1 lần, mỗi epoch lấy subset ngẫu nhiên)

In [ ]:
tokenizer = Tokenizer.from_file(f"{BASE_DIR}/tokenizer/tokenizer.json")
PAD_IDX = tokenizer.token_to_id('[PAD]')
BOS_IDX = tokenizer.token_to_id('[BOS]')
EOS_IDX = tokenizer.token_to_id('[EOS]')
print(f'Vocab size: {tokenizer.get_vocab_size():,}')

class TranslationDataset(Dataset):
    def __init__(self, path):
        print(f'Đang đọc {path} ...')
        with open(path, 'r', encoding='utf-8') as f:
            self.lines = f.readlines()
        print(f'✅ Tổng: {len(self.lines):,} câu')

    def __len__(self): return len(self.lines)

    def __getitem__(self, idx):
        parts = self.lines[idx].strip().split('\t')
        if len(parts) != 3:
            return torch.tensor([BOS_IDX, EOS_IDX]), torch.tensor([BOS_IDX, EOS_IDX])
        tag, src, tgt = parts
        s = tokenizer.encode(f'{tag} {src}').ids[:MAX_LEN - 2]
        t = tokenizer.encode(tgt).ids[:MAX_LEN - 2]
        return (torch.tensor([BOS_IDX] + s + [EOS_IDX], dtype=torch.long),
                torch.tensor([BOS_IDX] + t + [EOS_IDX], dtype=torch.long))

def collate_fn(batch):
    src, tgt = zip(*batch)
    return (pad_sequence(src, padding_value=PAD_IDX, batch_first=True),
            pad_sequence(tgt, padding_value=PAD_IDX, batch_first=True))

full_dataset = TranslationDataset(f"{BASE_DIR}/data/processed/train.txt")
TOTAL_SIZE   = len(full_dataset)

## 🚀 Training Loop

In [ ]:
VOCAB_SIZE = tokenizer.get_vocab_size()
enc   = EncoderLSTM(VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS, PAD_IDX)
dec   = DecoderLSTM(VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS, PAD_IDX)
model = Seq2Seq(enc, dec).to(DEVICE)

optimizer  = torch.optim.Adam(model.parameters(), lr=LR)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5, verbose=True)
criterion  = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
scaler     = GradScaler()
history    = {'epoch': [], 'train_loss': [], 'lr': []}
start_epoch = 1

# --- Auto-Resume ---
RESUME_CKPT = f"{OUT_DIR}/baseline_latest.pt"
if os.path.exists(RESUME_CKPT):
    print('🔄 Tìm thấy checkpoint, tiếp tục từ lần trước...')
    ckpt = torch.load(RESUME_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    history     = ckpt.get('history', history)
    start_epoch = ckpt['epoch'] + 1
    print(f'✅ Resume từ Epoch {ckpt["epoch"]}')
else:
    print('🆕 Bắt đầu training từ đầu.')

# =================== TRAINING LOOP ===================
print(f'\nBắt đầu huấn luyện LSTM | {start_epoch} → {EPOCHS} epochs | {SUBSET_SIZE:,} câu/epoch\n')

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()

    # Subset ngẫu nhiên cho epoch này
    indices = random.sample(range(TOTAL_SIZE), min(SUBSET_SIZE, TOTAL_SIZE))
    subset  = Subset(full_dataset, indices)
    loader  = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True,
                         collate_fn=collate_fn, num_workers=2, pin_memory=True)

    total_loss = 0
    optimizer.zero_grad()

    for i, (src, tgt) in enumerate(loader):
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)

        with autocast():
            output = model(src, tgt)
            loss   = criterion(output[:, 1:].reshape(-1, output.shape[2]),
                               tgt[:, 1:].reshape(-1))
            loss   = loss / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (i + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUM_STEPS

        if i % 500 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f'  Epoch {epoch:02d}/{EPOCHS} | '
                  f'Step {i:,}/{len(loader):,} | '
                  f'Loss: {loss.item()*ACCUM_STEPS:.4f} | '
                  f'LR: {current_lr:.2e}')

    avg_loss   = total_loss / len(loader)
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(avg_loss)   # ReduceLROnPlateau giảm LR khi loss không giảm

    history['epoch'].append(epoch)
    history['train_loss'].append(avg_loss)
    history['lr'].append(current_lr)

    print(f'\n✅ Epoch {epoch:02d}/{EPOCHS} DONE | '
          f'Avg Loss: {avg_loss:.4f} | LR: {current_lr:.2e}')

    # --- Lưu Checkpoint ---
    save_data = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'loss': avg_loss,
        'history': history
    }
    torch.save(save_data, f"{OUT_DIR}/baseline_latest.pt")          # Resume
    torch.save(save_data, f"{OUT_DIR}/baseline_ep{epoch:02d}.pt")   # Per-epoch
    with open(f"{OUT_DIR}/baseline_history.json", 'w') as f:
        json.dump(history, f)
    print(f'💾 Checkpoint saved (Epoch {epoch})\n' + '-'*60)

print('\n🎉 TRAINING HOÀN TẤT!')

## 📊 Biểu đồ kết quả (Chạy sau khi train xong)

In [ ]:
import json, matplotlib.pyplot as plt

hist_path = f"{OUT_DIR}/baseline_history.json"
with open(hist_path, 'r') as f:
    history = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('LSTM Baseline — Training Summary', color='white', fontsize=16, y=1.02)

styles = [
    ('train_loss', '#f59e0b', 'Training Loss', 'Loss'),
    ('lr',         '#ec4899', 'Learning Rate', 'LR'),
]

for ax, (key, color, title, ylabel) in zip(axes, styles):
    ax.set_facecolor('#1a1d2e')
    ax.plot(history['epoch'], history[key],
            color=color, linewidth=2.5, marker='o', markersize=7)
    ax.fill_between(history['epoch'], history[key], alpha=0.15, color=color)
    ax.set_title(title, color='white', fontsize=14, pad=10)
    ax.set_xlabel('Epoch', color='#94a3b8', fontsize=11)
    ax.set_ylabel(ylabel, color='#94a3b8', fontsize=11)
    ax.tick_params(colors='#94a3b8')
    ax.spines[:].set_color('#2d3748')
    ax.grid(True, alpha=0.2, color='#4a5568', linestyle='--')

    # Annotation giá trị nhỏ nhất
    min_val = min(history[key])
    min_ep  = history['epoch'][history[key].index(min_val)]
    ax.annotate(f'min={min_val:.4f}', xy=(min_ep, min_val),
                xytext=(min_ep + 0.5, min_val * 1.05),
                color=color, fontsize=9,
                arrowprops=dict(arrowstyle='->', color=color))

plt.tight_layout()
out_img = f"{OUT_DIR}/baseline_curve_final.png"
plt.savefig(out_img, dpi=150, facecolor=fig.get_facecolor(), bbox_inches='tight')
plt.show()
print(f'✅ Biểu đồ đã lưu: {out_img}')